# ChurnSense AI - Learning Lab

Run the production scripts first, then use this notebook to inspect each stage interactively. Read `docs/learning_guide.md` before each code section.

## 1. Load and inspect the official IBM dataset
Pandas reads the CSV into a DataFrame. `head`, `info`, and `describe` reveal schema, types, and distribution.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import load_raw_data, clean_data
raw = load_raw_data(ROOT / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
raw.shape, raw.head()

## 2. Clean and engineer features
Notice that `TotalCharges` changes from text to a numeric field, while `customerID` remains for lookup but is excluded from training.

In [ ]:
from src.features import add_features
df = add_features(clean_data(raw))
df[['TotalCharges', 'Churn', 'AvgMonthlySpend', 'TenureGroup', 'AutoPay']].head()

## 3. Explore churn hypotheses
The group mean of binary `Churn` is the group churn rate.

In [ ]:
df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)

## 4. Train and evaluate
Run the command below once; the function saves all models and comparison metrics.

In [ ]:
from src.train import train_all
results = train_all()
results

## 5. Explain and action results
SHAP explains feature influence on the selected model. The recommendation engine is separate, transparent business logic.

In [ ]:
from src.explain import create_shap_summary
from src.recommend import retention_recommendation
create_shap_summary()
sample = df.iloc[0]
retention_recommendation(sample, 0.72)